### Window Generation 

Window generation and data preparation for model ingestion. Benign and attack traffic are windowed into two separate sets. Each message is turned into a sliding window over consecutive messages. 

**Windows for Benign** would have window size: 32 and stride: 16 due to the volume of data. We will be ingesting three main files which contain 10M rows each. 

**Windows for Attack** would have window size: 32 and stride: 2 the attack files are smaller and the injection time is shorter, a small stride produces more windows for the model. 

The feature extractor should take a feature array with the shape [num_windows, window_size, num_features] and a target array with the shape [num_windows]. The return value should be the transformed feature and target arrays. 

Structure should follow (num_windows, window_size, features). 

#### Feature Vector 

Every message, benign or attack, is converted into the same 11-feature vector. Keeping one shared representation lets the model learn each attack's behaviour from the same inputs, and matches the features examined in the Attack EDA and Benign EDA. 

11 Features contain: [ CAN_ID , b0, b1, b2, b3, b4, b5, b6, b7, DLC, Δt ] 

- **CAN_ID** — identifier as an integer
- **b0–b7** — payload bytes, padded to 8 bytes
- **DLC** — data length code
- **Δt** — inter-arrival time since the previous message


In [1]:
import sys
from pathlib import Path
from collections import Counter
import pandas as pd
import torch

BASE = Path("/Users/anita/Documents/TFM/SSL_CyberSecurity")
sys.path.append(str(BASE / "src"))
from extract_can_fields import extract_can_fields 
from load_attack_file import load_attack_file, ATTACKS

In [2]:
BENIGN_FILES = [
    BASE / "Benign" / "Day_3" / "Benign_day3_file1.log",
    BASE / "Benign 2" / "Day_5" / "Benign_day5_file1.log",
    BASE / "Benign 2" / "Day_6" / "Benign_day6_file1.log",
]

WINDOW_SIZE = 32
STRIDE = 16
TRAIN_RATIO = 0.70

BENIGN_OUTPUT_DIR = BASE / "Benign_windows"
BENIGN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Benign files:", len(BENIGN_FILES))
print("Window size / stride:", WINDOW_SIZE, "/", STRIDE)
print("Train ratio:", TRAIN_RATIO)
print("Output directory:", BENIGN_OUTPUT_DIR)

Benign files: 3
Window size / stride: 32 / 16
Train ratio: 0.7
Output directory: /Users/anita/Documents/TFM/SSL_CyberSecurity/Benign_windows


In [3]:
ATTACK_WINDOW_SIZE = 32
ATTACK_STRIDE = 2
ATTACK_FILES = list(ATTACKS.keys())

ATTACK_OUTPUT_DIR = BASE / "Attack_windows"
ATTACK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Attack files:", len(ATTACK_FILES))
print(ATTACK_FILES)
print("Output directory:", ATTACK_OUTPUT_DIR)

Attack files: 9
['Steering_angle_attack', 'Brake_warning_attack', 'Power_steering_attack', 'Min_speedometer_attack_1', 'EMS_replay_attack', 'Steering_angle_replay', 'Fuzzing_random_IDs', 'Fuzzing_valid_IDs', 'DoS_attack']
Output directory: /Users/anita/Documents/TFM/SSL_CyberSecurity/Attack_windows


Process the message and extract the features to convert it into the 11 feature vector [CAN_ID] + 8 payload bytes + [DLC, Δt] = 11 features

In [4]:
def message_to_features(extracted, previous_timestamp): 
    can_id = int(extracted["can_id"], 16)
    
    payload_bytes = []
    for byte in extracted["payload"]:
        payload_bytes.append(int(byte, 16))
        
    timestamp = extracted["timestamp"]
    if previous_timestamp is None:
        delta_t = 0  # since the first message has no previous timestamp
    else:
        delta_t = timestamp - previous_timestamp
        
    features = [can_id] + payload_bytes + [extracted["dlc"], delta_t]
    return features, timestamp     

#### Processing Files

In [5]:
def load_message_features(file_path): 
    message_features = []
    previous_timestamp = None
    
    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            extracted = extract_can_fields(line)
            if extracted is None:
                continue

            features, previous_timestamp = message_to_features(extracted, previous_timestamp)
            message_features.append(features)
            
    return message_features 

In [6]:
def make_windows(message_features, window_size,stride):
    windows = []
    current_window = []
    for feature in message_features:
        current_window.append(feature)
        if len(current_window) == window_size:
            windows.append(current_window.copy())
            current_window = current_window[stride:]
            
    return torch.tensor(windows, dtype=torch.float32)

Three files will be used for the creation of the benign windows, which then will be stored as .pt in a folder on the root of the project. 

1. Benign_day3_file1.log 
2. Benign_day5_file1.log 
3. Benign_day6_file1.log 

Benign files are windowed separately, file by file. They are not concatenated before windowing, otherwise the end of one recording could share a window with the start of another, and the delta time between those messages would not be real bus timing. 

The 12 benign files behave very similarly, since they all contain the same 56 CAN IDs, message rates, and share the same frequent CAN IDs in almost identical proportions. This shows that the benign traffic is stable across the different recording days. 

Therefore the selected files are not more representative than the others, however they contain enough messages for reliable analysis and come from three different days, reducing the risk of analysing behaviour that is specific to one recording session. 

Every window is labelled Benign since there is no injection interval. 

In [ ]:
for file_path in BENIGN_FILES: 
    print("Processing File: ", file_path.name)
    
    message_features = load_message_features(file_path)
    
    # Split by time: first 70% for training the SSL model, last 30% for checking reconstruction
    split_index = int(len(message_features) * TRAIN_RATIO)
    
    train_windows = make_windows(message_features[:split_index], WINDOW_SIZE, STRIDE)
    test_windows = make_windows(message_features[split_index:], WINDOW_SIZE, STRIDE)
    
    train_path = BENIGN_OUTPUT_DIR / f"{file_path.stem}_train.pt"
    test_path = BENIGN_OUTPUT_DIR / f"{file_path.stem}_test.pt"
    
    # Every benign window is class 0, so labels are all zeros 
    torch.save({"features": train_windows, "labels": torch.zeros(len(train_windows), dtype=torch.long)}, train_path)
    torch.save({"features": test_windows, "labels": torch.zeros(len(test_windows), dtype=torch.long)}, test_path)
    
    print("train:", train_windows.shape, "| test:", test_windows.shape)
    print("Saved:", train_path.name, "and", test_path.name)
    
    
print("Done.")

Processing File:  Benign_day3_file1.log
train: torch.Size([429074, 32, 11]) | test: torch.Size([183888, 32, 11])
Saved: Benign_day3_file1_train.pt and Benign_day3_file1_test.pt
Processing File:  Benign_day5_file1.log
train: torch.Size([456547, 32, 11]) | test: torch.Size([195662, 32, 11])
Saved: Benign_day5_file1_train.pt and Benign_day5_file1_test.pt
Processing File:  Benign_day6_file1.log
train: torch.Size([449616, 32, 11]) | test: torch.Size([192692, 32, 11])
Saved: Benign_day6_file1_train.pt and Benign_day6_file1_test.pt
Done.


**load_message_features(file_path)** reads the whole log in time order and builds a long Python list. Each item is one message as an 11-number feature vector.


**split_index = int(len(message_features) * TRAIN_RATIO)** finds the cut point. If the file has 10 000 000 messages and TRAIN_RATIO is 0.70, then split_index is 7 000 000. Everything before that index is training traffic; everything from that index onward is reconstruction-test traffic.

**make_windows(message_features[:split_index], ...)**  slides windows only on the first 70%
**make_windows(message_features[split_index:], ...)** slides windows only on the last 30%

**labels:** a vector of zeros, same length as the number of windows (all benign), SSL pretraning uses only features and does not use these labels, mainly used for consistency with attack windows. 

Because the split happens before windowing, no window can contain messages from both sides.

When all three days are done, we have **6 .pt files: 3 for pretraining**, **3 for checking reconstruction**. Notebook 04 will load the _train.pt files to train the SSL model, and the _test.pt files only to measure reconstruction loss.

In [ ]:
# Sort the files by name to make it more organized
pt_files = sorted(BENIGN_OUTPUT_DIR.glob("*_train.pt")) + sorted(BENIGN_OUTPUT_DIR.glob("*_test.pt"))
print("Train/test files found:", len(pt_files))

rows = []
for pt_path in pt_files:
    data = torch.load(pt_path, map_location="cpu", weights_only=False)

    features = data["features"]
    labels = data["labels"]
    split_name = "train" if pt_path.name.endswith("_train.pt") else "test"

    rows.append({
        "file": pt_path.name,
        "split": split_name,
        "n_windows": features.shape[0],
        "window_shape": tuple(features.shape[1:]),
        "label_values": labels.unique().tolist(),
    })
pd.DataFrame(rows)

Train/test files found: 6


,file,split,n_windows,window_shape,label_values
0,Benign_day3_file1_train.pt,train,429074,"(32, 11)",[0]
1,Benign_day5_file1_train.pt,train,456547,"(32, 11)",[0]
2,Benign_day6_file1_train.pt,train,449616,"(32, 11)",[0]
3,Benign_day3_file1_test.pt,test,183888,"(32, 11)",[0]
4,Benign_day5_file1_test.pt,test,195662,"(32, 11)",[0]
5,Benign_day6_file1_test.pt,test,192692,"(32, 11)",[0]


The table proves each window has the 32 x 11 feature vector, with label values 0 representing benign, and the train files hold the 70% and the test files 30% matching the split. 

### Attack Windows 

Attack windows use the same 11 feature vector as benign traffic [ CAN_ID , b0, b1, b2, b3, b4, b5, b6, b7, DLC, Δt ]. 

Each attack recording stays separate: windows are created per source file, never across files. Mixing files would break the time order of the bus.

Message labels come from the metadata injection interval via the load_attack function, where rows inside [start,end] get the attack class and rows outside stay Benign. 

A window label is taken as the majority of the message labels in that window. Windows whose majority is Benign are discarded, so only windows dominated by an attack class are kept. A single attack message inside an otherwise benign window is not enough to label the window as attack. 

Saved tensors go to an Attack_windows/ folder, with the corresponding .pt window files saved. 

In [ ]:
rows = []

for name in ATTACK_FILES: 
    df = load_attack_file(name)
    message_labels = df["label"].tolist()
    
    total_windows = 0 
    mixed_windows = 0 
    pure_attack_windows = 0 
    pure_benign_windows = 0 
    
    # Iterate over the message labels in steps of ATTACK_STRIDE
    for start in range(0, len(message_labels) - ATTACK_WINDOW_SIZE + 1, ATTACK_STRIDE):
        # Get the labels for the current window
        window_labels = message_labels[start : start + ATTACK_WINDOW_SIZE]
        # Count the total number of windows
        total_windows += 1 
        
        unique_labels = set(window_labels) # Get the unique labels in the current window
        
        # Check if the window contains both attack and benign messages
        if len(unique_labels) > 1:
            mixed_windows += 1 
        elif "Benign" in unique_labels:
            pure_benign_windows += 1 
        else: 
            pure_attack_windows += 1 
            
    rows.append({
        "file": name, 
        "total_windows": total_windows, 
        "mixed_windows": mixed_windows, 
        "pure_attack_windows": pure_attack_windows, 
        "pure_benign_windows": pure_benign_windows, 
        "percentage_mixed": round(100 * mixed_windows / total_windows, 2),
    })

In [8]:
rows_df = pd.DataFrame(rows)
rows_df.head(10)

,file,total_windows,mixed_windows,pure_attack_windows,pure_benign_windows,percentage_mixed
0,Steering_angle_attack,184800,31,63079,121690,0.02
1,Brake_warning_attack,294281,32,136388,157861,0.01
2,Power_steering_attack,180584,31,60881,119672,0.02
3,Min_speedometer_attack_1,272787,32,146515,126240,0.01
4,EMS_replay_attack,162058,32,64363,97663,0.02
5,Steering_angle_replay,237889,31,101450,136408,0.01
6,Fuzzing_random_IDs,364488,31,150510,213947,0.01
7,Fuzzing_valid_IDs,128691,31,28563,100097,0.02
8,DoS_attack,170878,31,91940,78907,0.02


In order to see how much of the windows contain either attack or benign traffic, we need to analyse whether if we cut the stream into windows of size 32 and stride 2 we get: 

- total windows: how many windows exist 
- mixed windows: windows that contain benign and attack messages 
- pure attack: windows that contain only attack messages 
- pure benign: windows that contain only benign messages 
- percentage mixed: percentage of windows that are mixed, which is about 0.01% so almost every window is unambiguous, pure benign windows can be dropped from the attack dataset. 

### Max Argument 

We use a max argument to keep track of the windows who can represent the maximum value class, meaning that instead of assigning the label DoS to a window which only has 1 DoS out of 32 messages, we look at the what are the majority of the messages inside that window. 

So we assign the label according to this, and we drop the windows whose majority is Benign. 

We later save the (32,11) features + that class label. 

The model learns the correlation between the 11 features over 32 steps as well as each attack specific detection pattern. 

In [9]:

CLASS_TO_ID = {
    "DoS": 1,
    "Spoofing": 2,
    "Fuzzing": 3,
    "Replay": 4,
}

print("Output directory:", ATTACK_OUTPUT_DIR)
print("Class mapping:", CLASS_TO_ID)


Output directory: /Users/anita/Documents/TFM/SSL_CyberSecurity/Attack_windows
Class mapping: {'DoS': 1, 'Spoofing': 2, 'Fuzzing': 3, 'Replay': 4}


Attack windows are labelled 1–4 here: benign majority windows are dropped, and notebook 04 shifts these to 0 to 3 for the attack only classifier.

#### Processing Files 

In [ ]:
def majority_label(window_labels):
    # Count the occurrences of each label in the window
    label_counts = Counter(window_labels)
    # Return the label with the highest count
    return max(label_counts, key=label_counts.get)

def make_attack_windows(name, window_size, stride):
    df = load_attack_file(name)
    
    features = []
    for i in range(len(df)):
        # Convert the payload to a list of integers
        payload_bytes = [int(b, 16) for b in df["payload"].iloc[i]]
        # Create a feature vector for the current message
        features.append([df["id_int"].iloc[i]] + payload_bytes + [df["dlc"].iloc[i], df["dt"].iloc[i]])

    # Get the message labels for the current file
    message_labels = df["label"].tolist()

    windows, window_labels = [], []
    
    # Iterate over the features in steps of the stride
    for start in range(0, len(features) - window_size + 1, stride):
        # Get the majority label for the current window
        class_name = majority_label(message_labels[start:start + window_size])
        # If the majority is Benign, skip this window
        if class_name == "Benign":
            continue
        
        # Append the window and its label to the lists
        windows.append(features[start:start + window_size])
        # Append the window and its label to the lists
        window_labels.append(CLASS_TO_ID[class_name])
        
    return (torch.tensor(windows, dtype=torch.float32), torch.tensor(window_labels, dtype=torch.long),
)

In [ ]:
for name in ATTACK_FILES:
    print("Processing file:", name)

    windows, labels = make_attack_windows(name, ATTACK_WINDOW_SIZE, ATTACK_STRIDE)

    save_path = ATTACK_OUTPUT_DIR / f"{name}.pt"
    torch.save({"features": windows, "labels": labels}, save_path)

    print("Saved window file:", save_path.name)

print("All attack windows have been saved.")

Processing_file:  Steering_angle_attack.log
Saved window file: /Users/anita/Documents/TFM/SSL_CyberSecurity/Attack_windows/Steering_angle_attack.pt
Processing_file:  Brake_warning_attack.log
Saved window file: /Users/anita/Documents/TFM/SSL_CyberSecurity/Attack_windows/Brake_warning_attack.pt
Processing_file:  Power_steering_attack.log
Saved window file: /Users/anita/Documents/TFM/SSL_CyberSecurity/Attack_windows/Power_steering_attack.pt
Processing_file:  Min_speedometer_attack_1.log
Saved window file: /Users/anita/Documents/TFM/SSL_CyberSecurity/Attack_windows/Min_speedometer_attack_1.pt
Processing_file:  EMS_replay_attack.log
Saved window file: /Users/anita/Documents/TFM/SSL_CyberSecurity/Attack_windows/EMS_replay_attack.pt
Processing_file:  Steering_angle_replay.log
Saved window file: /Users/anita/Documents/TFM/SSL_CyberSecurity/Attack_windows/Steering_angle_replay.pt
Processing_file:  Fuzzing_random_IDs.log
Saved window file: /Users/anita/Documents/TFM/SSL_CyberSecurity/Attack_wind

In [12]:
pt_files = sorted(ATTACK_OUTPUT_DIR.glob("*.pt"))
print("Files found:", len(pt_files))

rows = []
for pt_path in pt_files:
    data = torch.load(pt_path, map_location="cpu")

    features = data["features"]
    labels = data["labels"]

    rows.append({
        "file": pt_path.name,
        "keys": list(data.keys()),
        "features_shape": tuple(features.shape),
        "labels_shape": tuple(labels.shape),
        "dtype": str(features.dtype),
        "label_values": labels.unique().tolist(),
    })
    
pd.DataFrame(rows)

Files found: 9


,file,keys,features_shape,labels_shape,dtype,label_values
0,Brake_warning_attack.pt,"[features, labels]","(136404, 32, 11)","(136404,)",torch.float32,[2]
1,DoS_attack.pt,"[features, labels]","(91955, 32, 11)","(91955,)",torch.float32,[1]
2,EMS_replay_attack.pt,"[features, labels]","(64379, 32, 11)","(64379,)",torch.float32,[4]
3,Fuzzing_random_IDs.pt,"[features, labels]","(150525, 32, 11)","(150525,)",torch.float32,[3]
4,Fuzzing_valid_IDs.pt,"[features, labels]","(28579, 32, 11)","(28579,)",torch.float32,[3]
5,Min_speedometer_attack_1.pt,"[features, labels]","(146531, 32, 11)","(146531,)",torch.float32,[2]
6,Power_steering_attack.pt,"[features, labels]","(60896, 32, 11)","(60896,)",torch.float32,[2]
7,Steering_angle_attack.pt,"[features, labels]","(63094, 32, 11)","(63094,)",torch.float32,[2]
8,Steering_angle_replay.pt,"[features, labels]","(101466, 32, 11)","(101466,)",torch.float32,[4]
